In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn utilities for preprocessing and pipeline management
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Machine Learning Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Evaluation Metrics
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

# Link to the root directory to import our custom data loader module
sys.path.append(os.path.abspath(os.path.join('..')))
from src.data_loader import load_insurance_data

# Load the dataset
df = load_insurance_data()

# Inspect the target column distribution to see how imbalanced our data is
print("Target Variable Distribution ('Claimed'):")
print(df['Claimed'].value_counts(normalize=True))
print(f"\nDataset loaded successfully with shape: {df.shape}")

✅ Successfully loaded dataset from: ..\data\insurance_data.csv
Target Variable Distribution ('Claimed'):
Claimed
False    0.8465
True     0.1535
Name: proportion, dtype: float64

Dataset loaded successfully with shape: (10000, 21)


In [2]:
# 1. Define your Features (X) and Target (y)
# We exclude unique keys or leakage columns like CustomerID, ZipCode, ClaimAmount, TotalClaims, and TransactionDate
feature_cols = ['Age', 'Gender', 'Province', 'VehicleType', 'AnnualIncome', 'RiskScore', 'AnnualPremium', 'Deductible', 'NCD']
X = df[feature_cols]
y = df['Claimed']

# 2. Separate column names by data type
numerical_cols = ['Age', 'AnnualIncome', 'RiskScore', 'AnnualPremium', 'Deductible', 'NCD']
categorical_cols = ['Gender', 'Province', 'VehicleType']

# 3. Create preprocessing steps for both numeric and categorical features
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())  # Centers and scales numbers
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  # Converts text categories into numeric vectors
])

# 4. Combine transformers into a single preprocessor component
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 5. Split the data into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (8000, 9)
Testing set shape: (2000, 9)


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

print("--- Step 1: Loading and Preprocessing Dataset ---")

# 1. Load the raw insurance dataset
df = pd.read_csv(r"..\data\insurance_data.csv")

# 2. Separate your target indicator from your prediction features
# (Assuming 'Claimed' is your True/False column name)
X = df.drop(columns=['Claimed'])
y = df['Claimed']

# 3. THE FIX: Convert all categorical text strings (like 'Male') into numeric 0s and 1s
# 'drop_first=True' prevents multi-collinearity by dropping redundant columns 
X_encoded = pd.get_dummies(X, drop_first=True)

# 4. Split the encoded numeric features into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset loaded successfully with base shape: {df.shape}")
print(f"Encoded Training feature matrix shape: {X_train.shape}")
print(f"Encoded Testing feature matrix shape: {X_test.shape}")
print("\nTarget Variable Distribution ('Claimed'):")
print(y.value_counts(normalize=True))

--- Step 1: Loading and Preprocessing Dataset ---


Dataset loaded successfully with base shape: (10000, 21)
Encoded Training feature matrix shape: (8000, 10582)
Encoded Testing feature matrix shape: (2000, 10582)

Target Variable Distribution ('Claimed'):
Claimed
False    0.8465
True     0.1535
Name: proportion, dtype: float64


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("\n--- Step 2: Training Random Forest Classifier ---")

# 1. Initialize the model with balanced class weights to handle the true target imbalance
rf_model = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced', 
    random_state=42,
    max_depth=10  # Keeps the model robust and prevents over-fitting
)

# 2. Train the model using your clean, non-leaky datasets
rf_model.fit(X_train, y_train)
print("Model training complete!")

# 3. Predict classifications against the unseen test vectors
y_pred = rf_model.predict(X_test)

# 4. Print out your true comprehensive evaluation table
print("\n MODEL PERFORMANCE REPORT:")
print("==================================================")
print(classification_report(y_test, y_pred))
print("==================================================")

# 5. Extract and display legitimate predictive features
importances = rf_model.feature_importances_
feature_imp_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\n🔑 TRUE TOP 5 PREDICTIVE INSURANCE FEATURES:")
print(feature_imp_df.head(5).to_string(index=False))


--- Step 2: Training Random Forest Classifier ---
Model training complete!

📊 HONEST MODEL PERFORMANCE REPORT:
              precision    recall  f1-score   support

       False       0.91      0.81      0.86      1693
        True       0.35      0.56      0.43       307

    accuracy                           0.77      2000
   macro avg       0.63      0.69      0.65      2000
weighted avg       0.83      0.77      0.79      2000


🔑 TRUE TOP 5 PREDICTIVE INSURANCE FEATURES:
      Feature  Importance
          NCD    0.100826
 TotalPremium    0.087804
AnnualPremium    0.076368
   PastClaims    0.068744
          Age    0.065369
